In [1]:
import torch
import os
# Set CUDA memory management before importing torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import torch.optim as optim
from tqdm import tqdm
import data as Data
import models as Model
import torch.nn as nn
import argparse
import logging
import core.logger as Logger
from core.utils import *
import numpy as np
from misc.metric_tools import ConfuseMatrixMeter
from models.loss import *
from collections import OrderedDict
import core.metrics as Metrics
from misc.torchutils import get_scheduler, save_network
import wandb
import matplotlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datetime import datetime
from itertools import islice

In [2]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--config', type=str, default='/home/saraashojaeii/git/BuildingCD_mamba_based/config/second_cdmamba/second_cdmamba.json')
parser.add_argument('--phase', type=str, default='train', choices=['train', 'test'])
parser.add_argument('--model', type=str, default='')
parser.add_argument('--dataset', type=str, default='')
parser.add_argument('--tag', type=str, default='')
parser.add_argument('--seed', type=int, default=None)
parser.add_argument('--max_train_batches', type=int, default=0)
parser.add_argument('--max_val_batches', type=int, default=0)
parser.add_argument('--max_test_batches', type=int, default=0)
parser.add_argument('--change_threshold', type=float, default=0.2)

# Pass arguments explicitly as if they were typed on CLI
notebook_args = [
    "--config", "/workspace/BuildingCD_mamba_based/config/second_cdmamba/second_cdmamba.json",
    "--phase", "train",
    "--change_threshold", "0.5",
    "--seed", "42",
    "--tag", "debug",
    "--max_train_batches", "1",
    "--max_val_batches", "1",
    "--max_test_batches", "1"
]
args = parser.parse_args(notebook_args)

opt = Logger.parse(args)

opt = Logger.dict_to_nonedict(opt)

In [3]:
# Create a unique timestamped experiment subfolder for logs/results/checkpoints

exp_timestamp = datetime.now().strftime('%m%d_%H')
exp_name = opt.get('name', 'experiment')
dataset_suffix = getattr(args, 'dataset', None) or ''
tag_suffix = getattr(args, 'tag', None) or ''
suffix_parts = []
if dataset_suffix:
    suffix_parts.append(str(dataset_suffix))
if tag_suffix:
    suffix_parts.append(str(tag_suffix))
suffix = '_'.join(suffix_parts)
if suffix:
    exp_folder = f"{suffix}_{exp_timestamp}"
else:
    exp_folder = f"{exp_timestamp}"
for k in ['log', 'result', 'checkpoint']:
    if k in opt['path_cd'] and isinstance(opt['path_cd'][k], str):
        base_dir = opt['path_cd'][k]
        stamped = os.path.join(base_dir, exp_folder)
        opt['path_cd'][k] = stamped
        os.makedirs(stamped, exist_ok=True)
        
# Keep the subfolder name for reference
opt['path_cd']['exp_folder'] = exp_folder

# Assuming args already has a seed value
set_seed(args.seed if args.seed is not None else 42)  # fallback to 42 if not provided


[INFO] Random seed set to 42


In [4]:
#logging
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

Logger.setup_logger(logger_name=None, root=opt['path_cd']['log'], phase='train',
                    level=logging.INFO, screen=True)
Logger.setup_logger(logger_name='val', root=opt['path_cd']['log'], phase='val',
                    level=logging.INFO)
Logger.setup_logger(logger_name='test', root=opt['path_cd']['log'], phase='test',
                    level=logging.INFO)
logger = logging.getLogger('base')
logger.info(Logger.dict2str(opt))

# Set device with comprehensive debugging
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Using device: {device}')

# Initialize wandb only on main process
if opt.get('wandb') and opt['wandb'].get('project'):
    # Compose run name with dataset/tag suffixes for wandb as well
    run_name = exp_folder
    wandb.init(project=opt['wandb']['project'], config=opt, name=run_name)
    try:
        if hasattr(wandb, 'run') and wandb.run is not None:
            wandb.run.name = run_name
    except Exception:
        pass
else:
    wandb.init(mode="disabled")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(args.seed if args.seed is not None else 42)

#dataset
for phase, dataset_opt in opt['datasets'].items(): #train train{}
    #print(" phase is {}, dataopt is {}".format(phase, dataset_opt))
    if phase == 'train' and args.phase != 'test':
        print("Creat [train] change-detection dataloader")
        train_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        train_loader = Data.create_cd_dataloader(train_set, dataset_opt, phase, seed_worker, g)
        opt['len_train_dataloader'] = len(train_loader)

    elif phase == 'val' and args.phase != 'test':
        print("Creat [val] change-detection dataloader")
        val_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        val_loader = Data.create_cd_dataloader(val_set, dataset_opt, phase)
        opt['len_val_dataloader'] = len(val_loader)

    # elif phase == 'test' and args.phase == 'test':
    elif phase == 'test':
        print("Creat [test] change-detection dataloader")
        test_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        test_loader = Data.create_cd_dataloader(test_set, dataset_opt, phase)
        opt['len_test_dataloader'] = len(test_loader)

logger.info('Initial Dataset Finished')

#Create cd model
cd_model = Model.create_CD_model(opt)

# Initialize model weights to prevent NaN loss - more conservative
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.xavier_normal_(m.weight, gain=0.1)  # Very small gain
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.001)  # Very small std
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

cd_model.apply(init_weights)
cd_model.to(device)
logger.info(f'CD Model moved to device: {device}')

# Verify model is actually on GPU
if torch.cuda.is_available():
    model_device = next(cd_model.parameters()).device
    logger.info(f'Model parameters are on device: {model_device}')
    if model_device.type != 'cuda':
        logger.error('WARNING: Model parameters are NOT on GPU!')
    else:
        logger.info('✓ Model successfully moved to GPU')

# Enable gradient checkpointing if available to save memory
if hasattr(cd_model, 'gradient_checkpointing_enable'):
    cd_model.gradient_checkpointing_enable()

num_classes = opt['model']['n_classes']
logger.info(f"Number of classes for loss function: {num_classes}")

#Create criterion (segmentation losses use semantic num_classes; change head will use 2)
if opt['model']['loss'] == 'ce_dice':
    loss_fun = CEDiceLoss(num_classes=num_classes)
    loss_fun_change = CEDiceLoss(num_classes=2)
elif opt['model']['loss'] == 'ce':
    # CrossEntropy can be used as a function or nn.Module. Using function for now.
    loss_fun = cross_entropy_loss_fn
    loss_fun_change = cross_entropy_loss_fn
elif opt['model']['loss'] == 'dice':
    loss_fun = DiceOnlyLoss(num_classes=num_classes)
    loss_fun_change = DiceOnlyLoss(num_classes=2)
elif opt['model']['loss'] == 'extended_triplet':
    # Extended multi-task loss: seg(t1)+seg(t2)+change + cross-time consistency + coupling
    base_seg = CEDiceLoss(num_classes=num_classes)
    cfg = opt['model'].get('extended_triplet', {})
    loss_fun = TripletChangeSegLoss(
        seg_loss_fn=base_seg,
        lambda_seg=cfg.get('lambda_seg', 1.0),
        lambda_cd=cfg.get('lambda_cd', 1.0),
        lambda_unch=cfg.get('lambda_unch', 0.2),
        lambda_ch=cfg.get('lambda_ch', 0.2),
        lambda_cpl=cfg.get('lambda_cpl', 0.5),
        T=cfg.get('T', 4.0),
        margin=cfg.get('margin', 0.3)
    )
else:
    raise ValueError(f"Unsupported loss function type: {opt['model']['loss']}")

# If losses are nn.Module, move them to the device
if isinstance(loss_fun, nn.Module):
    loss_fun.to(device)
elif 'loss_fun_change' in locals() and isinstance(loss_fun_change, nn.Module):
    loss_fun_change.to(device)
# Fallback: if loss_fun_change wasn't defined (e.g., for unsupported options), reuse loss_fun
elif 'loss_fun_change' not in locals():
    loss_fun_change = loss_fun

#Create optimizer
if opt['train']["optimizer"]["type"] == 'adam':
    optimer = optim.Adam(cd_model.parameters(), lr=opt['train']["optimizer"]["lr"])
elif opt['train']["optimizer"]["type"] == 'adamw':
    optimer = optim.AdamW(cd_model.parameters(), lr=opt['train']["optimizer"]["lr"])
elif opt['train']["optimizer"]["type"] == 'sgd':
    optimer = optim.SGD(cd_model.parameters(), lr=opt['train']["optimizer"]["lr"],
                        momentum=0.9, weight_decay=5e-4)

# Initialize mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

metric = ConfuseMatrixMeter(n_class=2)  # For binary change detection (change/no-change)
log_dict = OrderedDict()

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.8)  # if you really want this

25-08-25 16:47:22.936 - INFO:   name: SECOND-train-CDMamba
  phase: train
  gpu_ids: [0]
  wandb:[
    project: BuildingCD_mamba_based
  ]
  path_cd:[
    log: /root/home/pvc/Building_changedetection_job/experiments/logs/debug_0825_16
    result: /root/home/pvc/Building_changedetection_job/experiments/results/debug_0825_16
    checkpoint: /root/home/pvc/Building_changedetection_job/experiments/checkpoint/debug_0825_16
    resume_state: None
    experiments_root: experiments/SECOND-train-CDMamba_250825_164719
    exp_folder: debug_0825_16
  ]
  datasets:[
    train:[
      name: SECOND-CD-256
      datasetroot: /root/home/pvc/SECOND/train
      resolution: 512
      num_workers: 4
      batch_size: 2
      use_shuffle: True
      data_len: -1
    ]
    val:[
      name: SECOND-CD-256
      datasetroot: /root/home/pvc/SECOND/val
      resolution: 512
      num_workers: 4
      batch_size: 2
      use_shuffle: True
      data_len: -1
    ]
    test:[
      name: SECOND-CD-256
      datase

/root/home/pvc/Building_changedetection_job/experiments/logs/debug_0825_16/train.log
/root/home/pvc/Building_changedetection_job/experiments/logs/debug_0825_16/val.log
/root/home/pvc/Building_changedetection_job/experiments/logs/debug_0825_16/test.log


25-08-25 16:47:23.350 - INFO: Using device: cuda
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:


Abort: 